In [ ]:
# 필요한 라이브러리들을 임포트합니다.
import torch  # PyTorch 딥러닝 프레임워크
from torch.utils.data import Dataset, DataLoader  # 데이터 처리를 위한 클래스들
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW  # Hugging Face 트랜스포머 라이브러리
from sklearn.model_selection import train_test_split  # 데이터 분할을 위한 함수
from sklearn.metrics import classification_report  # 분류 모델 평가를 위한 함수
import numpy as np  # 수치 연산을 위한 라이브러리

# 커스텀 데이터셋 클래스를 정의합니다.
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts  # 입력 텍스트 리스트
        self.labels = labels  # 해당 텍스트의 라벨 리스트
        self.tokenizer = tokenizer  # 텍스트를 토큰화할 토크나이저
        self.max_len = max_len  # 최대 시퀀스 길이

    def __len__(self):
        return len(self.texts)  # 데이터셋의 총 샘플 수를 반환

    def __getitem__(self, item):
        text = str(self.texts[item])  # 현재 인덱스의 텍스트
        label = self.labels[item]  # 현재 인덱스의 라벨

        # 토크나이저를 사용하여 텍스트를 인코딩합니다.
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,  # [CLS], [SEP] 같은 특수 토큰 추가
            max_length=self.max_len,  # 최대 길이 설정
            return_token_type_ids=False,  # token_type_ids 반환하지 않음
            padding='max_length',  # 패딩 적용
            truncation=True,  # 최대 길이를 초과하는 경우 자르기
            return_attention_mask=True,  # attention mask 반환
            return_tensors='pt',  # PyTorch 텐서 형태로 반환
        )

        # 필요한 정보를 딕셔너리 형태로 반환
        return {
            'input_ids': encoding['input_ids'].flatten(),  # 입력 시퀀스
            'attention_mask': encoding['attention_mask'].flatten(),  # 어텐션 마스크
            'labels': torch.tensor(label, dtype=torch.long)  # 라벨
        }

# 모델과 토크나이저를 로드합니다.
tokenizer = AutoTokenizer.from_pretrained("monologg/kobert")  # KoBERT 토크나이저 로드
model = AutoModelForSequenceClassification.from_pretrained("monologg/kobert", num_labels=3)  # KoBERT 모델 로드, 3개 클래스 분류 설정

# 예시 데이터를 생성합니다. (실제 사용 시에는 이 부분을 실제 데이터로 대체해야 합니다)
texts = ["관광지 추천해주세요", "버스 시간표 알려주세요", "근처 병원 어디있나요"] * 1000
labels = [0, 1, 2] * 1000  # 0: 관광, 1: 교통, 2: 건강

# 데이터를 학습 세트와 검증 세트로 분할합니다.
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2)

# 학습 및 검증 데이터셋을 생성합니다.
train_dataset = TextClassificationDataset(train_texts, train_labels, tokenizer, max_len=128)
val_dataset = TextClassificationDataset(val_texts, val_labels, tokenizer, max_len=128)

# DataLoader를 생성합니다. 이는 모델 학습 시 배치 단위로 데이터를 제공합니다.
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# 한 에폭 동안의 학습을 수행하는 함수를 정의합니다.
def train_epoch(model, data_loader, optimizer, device):
    model.train()  # 모델을 학습 모드로 설정
    losses = []
    for batch in data_loader:
        optimizer.zero_grad()  # 그래디언트 초기화
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        losses.append(loss.item())  # 손실 값 저장
        loss.backward()  # 역전파 수행
        optimizer.step()  # 옵티마이저로 가중치 업데이트
    return np.mean(losses)  # 평균 손실 반환

# 모델을 평가하는 함수를 정의합니다.
def eval_model(model, data_loader, device):
    model.eval()  # 모델을 평가 모드로 설정
    predictions = []
    actual_labels = []
    with torch.no_grad():  # 그래디언트 계산 비활성화
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs.logits, dim=1)  # 가장 높은 확률의 클래스 선택
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())
    return classification_report(actual_labels, predictions)  # 분류 보고서 반환

# 학습에 사용할 디바이스를 설정합니다 (GPU 사용 가능시 GPU, 아니면 CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)  # 모델을 해당 디바이스로 이동

# 옵티마이저를 정의합니다.
optimizer = AdamW(model.parameters(), lr=2e-5)

# 학습을 실행합니다.
epochs = 3
for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, optimizer, device)
    print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}')
    print(eval_model(model, val_loader, device))  # 각 에폭 후 모델 평가

# 학습된 모델을 저장합니다.
torch.save(model.state_dict(), 'kobert_classifier.pth')